# Análise Exploratória (EDA) - O Algoritmo do Sucesso
Neste notebook, vamos analisar os dados salvos no nosso banco SQLite para descobrir o que faz uma música virar um hit no Spotify.

_Dica: Clique na célula abaixo e aperte Shift + Enter para rodar o código._

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3

# 1. Conectar ao banco de dados
conn = sqlite3.connect('../spotify_brasil.db')

# 2. Puxar todas as faixas e seus atributos
query = '''
SELECT f.nome, f.popularidade, f.genero, a.danceability, a.energy, a.valence, a.tempo
FROM faixas f
JOIN atributos_audio a ON f.id = a.faixa_id
'''
df = pd.read_sql(query, conn)
df.head()

## 1. O DNA do Hit (Energia vs Popularidade)
Vamos criar um gráfico de distribuição separando o que é considerado um Hit (popularidade acima de 70) do restante das músicas.

In [ ]:
# Cria uma nova coluna separando os Hits
df['is_hit'] = df['popularidade'] >= 70

plt.figure(figsize=(10, 6))
sns.histplot(data=df, x='energy', hue='is_hit', bins=30, kde=True)
plt.title('Distribuição de Energia: Hits (Laranja) vs Resto (Azul)')
plt.show()

## 2. Machine Learning: Prevendo um Hit
Vamos treinar um modelo preditivo rápido (Random Forest) para ver qual atributo musical tem maior peso para definir se uma música entra no Top Charts.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

features = ['energy', 'danceability', 'valence', 'tempo']
df_ml = df.dropna(subset=features + ['is_hit']).copy()

X = df_ml[features]
y = df_ml['is_hit']

# Separa os dados de treino e de teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Treina a inteligência artificial
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Analisa qual característica foi mais decisiva para o sucesso
importances = pd.Series(model.feature_importances_, index=features).sort_values()
importances.plot(kind='barh', color='green', title='Importância de cada atributo para o Hit')
plt.show()